In [8]:
import pandas as pd
import requests
import json
import time

In [9]:
API_URL = "https://datasets-server.huggingface.co/rows"

dataset = "nik-55/bhagavad-gita"
config = "conversations"
split = "train"

all_rows = []
offset = 0
length = 100

while True:

    params = {
        "dataset": dataset,
        "config": config,
        "split": split,
        "offset": offset,
        "length": length
    }

    response = requests.get(API_URL, params=params)
    response.raise_for_status()

    data = response.json()

    rows = data["rows"]

    if not rows:
        break

    all_rows.extend(rows)

    print(f"Downloaded {len(all_rows)} rows")

    offset += length

    if offset >= data["num_rows_total"]:
        break

    time.sleep(0.2)


# Save complete API response rows
with open("bhagavad_gita_sft_raw.json", "w", encoding="utf-8") as f:
    json.dump(all_rows, f, ensure_ascii=False, indent=2)

print(f"\nTotal rows downloaded: {len(all_rows)}")
print("Saved to: bhagavad_gita_sft_raw.json")

Downloaded 100 rows
Downloaded 200 rows
Downloaded 300 rows
Downloaded 400 rows
Downloaded 500 rows
Downloaded 600 rows
Downloaded 620 rows

Total rows downloaded: 620
Saved to: bhagavad_gita_sft_raw.json


In [15]:
import json

# Read the downloaded file
with open("C:\\Users\\Tvari\\Desktop\\TvaritRepo\\SLM\\slm-from-scratch\\data\\bhagavad_gita_sft_raw.json", "r", encoding="utf-8") as f:
    data = json.load(f)

# Keep only messages
cleaned_data = []

for item in data:
    cleaned_data.append({
        "messages": item["row"]["messages"]
    })

# Save cleaned data
with open("bhagavad_gita_sft_clean.json", "w", encoding="utf-8") as f:
    json.dump(cleaned_data, f, ensure_ascii=False, indent=2)

print("Total conversations:", len(cleaned_data))
print("Saved successfully!")

Total conversations: 620
Saved successfully!


In [11]:
import os

print(os.path.exists("bhagavad_gita_sft_raw.json"))
print(os.path.getsize("bhagavad_gita_sft_raw.json"))

True
2244198


### Fine Tuning started from here 

In [12]:
import json
from collections import Counter

with open("bhagavad_gita_sft_clean.json", "r", encoding="utf-8") as f:
    data = json.load(f)

role_counts = Counter()
message_lengths = []

for conversation in data:
    messages = conversation.get("messages", [])

    for message in messages:
        role_counts[message.get("role")] += 1
        message_lengths.append(len(message.get("content", "")))

print("Total conversations:", len(data))
print("Roles:", role_counts)
print("Shortest message:", min(message_lengths))
print("Longest message:", max(message_lengths))

Total conversations: 620
Roles: Counter({'user': 1120, 'assistant': 1114})
Shortest message: 12
Longest message: 3397


In [17]:
for conversation in data[:5]:
    print(json.dumps(conversation, ensure_ascii=False, indent=2))
    print("=" * 80)

{
  "row_idx": 0,
  "row": {
    "messages": [
      {
        "content": "The passage highlights that our problems are often not unique and that solutions, much like a 'senpai' or guide, already exist. It prompts us to seek someone who is a few steps ahead. How does one truly recognize such a guide, especially in spiritual matters, and what is the profound significance of submitting to their wisdom in our journey of self-improvement and finding direction, as taught in the Bhagavad Gita?",
        "role": "user"
      },
      {
        "content": "Indeed, the wisdom in seeking guidance is profound and timeless. The Bhagavad Gita emphasizes the crucial role of a spiritual teacher or 'guru' in illuminating our path, much like your 'senpai.' Lord Krishna Himself, in His role as the ultimate guide to Arjuna, lays out the process for acquiring true knowledge. He states in Bhagavad Gita 4.34:तद्विद्धि प्रणिपातेन परिप्रश्नेन सेवया। उपदेक्ष्यन्ति ते ज्ञानं ज्ञानिनस्तत्त्वदर्शिनः।।(Tad viddhi 

In [19]:
cleaned_conversations = []

for conversation in data:
    messages = conversation.get("messages")

    if messages is None:
        messages = conversation.get("row", {}).get("messages", [])

    cleaned_messages = []

    for message in messages:
        content = message.get("content")

        if not isinstance(content, str):
            continue

        content = content.strip()

        if not content:
            continue

        cleaned_messages.append({
            "role": message.get("role"),
            "content": content
        })

    if cleaned_messages:
        cleaned_conversations.append({"messages": cleaned_messages})

cleaned_data = cleaned_conversations
print("Total cleaned conversations:", len(cleaned_data))

Total cleaned conversations: 620


In [21]:
for conversation in cleaned_data[:1]:
    print(json.dumps(conversation, ensure_ascii=False, indent=2))
    print("=" * 80)

{
  "messages": [
    {
      "role": "user",
      "content": "The passage highlights that our problems are often not unique and that solutions, much like a 'senpai' or guide, already exist. It prompts us to seek someone who is a few steps ahead. How does one truly recognize such a guide, especially in spiritual matters, and what is the profound significance of submitting to their wisdom in our journey of self-improvement and finding direction, as taught in the Bhagavad Gita?"
    },
    {
      "role": "assistant",
      "content": "Indeed, the wisdom in seeking guidance is profound and timeless. The Bhagavad Gita emphasizes the crucial role of a spiritual teacher or 'guru' in illuminating our path, much like your 'senpai.' Lord Krishna Himself, in His role as the ultimate guide to Arjuna, lays out the process for acquiring true knowledge. He states in Bhagavad Gita 4.34:तद्विद्धि प्रणिपातेन परिप्रश्नेन सेवया। उपदेक्ष्यन्ति ते ज्ञानं ज्ञानिनस्तत्त्वदर्शिनः।।(Tad viddhi praṇipātena pa

In [22]:
import re
import unicodedata
import json


def clean_text(text):
    # Remove Devanagari / Sanskrit script
    text = re.sub(r'[\u0900-\u097F]+', '', text)

    # Convert Romanized Sanskrit characters to normal ASCII
    # Example: praṇipātena -> pranipatena
    text = unicodedata.normalize("NFKD", text)
    text = "".join(
        char for char in text
        if not unicodedata.combining(char)
    )

    # Remove extra spaces
    text = re.sub(r'[ \t]+', ' ', text)

    # Remove too many blank lines
    text = re.sub(r'\n{3,}', '\n\n', text)

    return text.strip()


final_cleaned_data = []

for conversation in cleaned_data:

    new_messages = []

    for message in conversation["messages"]:

        cleaned_content = clean_text(message["content"])

        # Skip empty messages
        if not cleaned_content:
            continue

        new_messages.append({
            "role": message["role"],
            "content": cleaned_content
        })

    # Keep only non-empty conversations
    if new_messages:
        final_cleaned_data.append({
            "messages": new_messages
        })


print("Before cleaning:", len(cleaned_data))
print("After cleaning:", len(final_cleaned_data))

Before cleaning: 620
After cleaning: 620


In [26]:
with open("bhagavad_gita_sft_no_sanskrit.json", "w", encoding="utf-8") as f:
    json.dump(
        final_cleaned_data,
        f,
        ensure_ascii=False,
        indent=2
    )

print("Saved successfully!")

Saved successfully!


In [ ]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.decoders import ByteLevel as ByteLevelDecoder


def build_fast_tokenizer(merges, vocab):
    byte_values = (
        list(range(ord("!"), ord("~") + 1))
        + list(range(ord("¡"), ord("¬") + 1))
        + list(range(ord("®"), ord("ÿ") + 1))
    )

    unicode_values = byte_values.copy()
    extra_index = 0

    for byte_value in range(256):
        if byte_value not in byte_values:
            byte_values.append(byte_value)
            unicode_values.append(256 + extra_index)
            extra_index += 1

    byte_encoder = {
        byte_value: chr(unicode_value)
        for byte_value, unicode_value
        in zip(byte_values, unicode_values)
    }

    def bytes_to_token_string(byte_sequence):
        return "".join(
            byte_encoder[byte_value]
            for byte_value in byte_sequence
        )

    fast_vocab = {
        bytes_to_token_string(byte_sequence): token_id
        for token_id, byte_sequence in vocab.items()
    }

    fast_merges = [
        (
            bytes_to_token_string(vocab[first_id]),
            bytes_to_token_string(vocab[second_id]),
        )
        for first_id, second_id in merges
    ]

    tokenizer = Tokenizer(
        BPE(
            vocab=fast_vocab,
            merges=fast_merges,
        )
    )

    tokenizer.pre_tokenizer = ByteLevel(
        add_prefix_space=False,
        use_regex=False,
    )

    tokenizer.decoder = ByteLevelDecoder()

    return tokenizer


fast_tokenizer = build_fast_tokenizer(
    merges=merges,
    vocab=vocab,
)

print("Fast tokenizer created")
print("Vocabulary size:", fast_tokenizer.get_vocab_size())

In [25]:
lengths = []
unk_counts = []

for conversation in final_cleaned_data:
    text = ""

    for msg in conversation["messages"]:
        text += f"{msg['role']}: {msg['content']}\n"

    ids = tokenizer.encode(text)

    lengths.append(len(ids))

    if tokenizer.unk_token_id is not None:
        unk_counts.append(ids.count(tokenizer.unk_token_id))

print("Number of conversations:", len(lengths))
print("Minimum tokens:", min(lengths))
print("Maximum tokens:", max(lengths))
print("Average tokens:", sum(lengths) / len(lengths))

if unk_counts:
    print("Total UNK tokens:", sum(unk_counts))
    print("Conversations containing UNK:",
          sum(x > 0 for x in unk_counts))

NameError: name 'tokenizer' is not defined